In [ ]:
!pip install -q huggingface_hub datasets torch tqdm

In [12]:
import os
import sys
import torch
import json
import pickle
from torch.utils.data import DataLoader
from huggingface_hub import HfApi, hf_hub_download

In [7]:
import os
import sys
current_dir = os.getcwd()
src_path = os.path.join(current_dir, 'src') if 'src' in os.listdir() else os.path.join(current_dir, 'n-gram-modeling', 'src')
sys.path.append(src_path)
print(f"added to path: {src_path}")

fatal: destination path 'n-gram-modeling' already exists and is not an empty directory.
Already up to date.


In [8]:
from data import get_jefferson_text, preprocess, build_vocab, TrigramDataset
from models import CountTrigramModel, NeuralTrigramModel
from train import train_neural_model
from decode import generate_greedy, generate_top_k, generate_nucleus, generate_beam_search
from eval import calculate_perplexity_count, calculate_perplexity_neural

In [ ]:
text = get_jefferson_text()
tokens = preprocess(text)

# Split into train and test
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

vocab, word2idx, idx2word = build_vocab(train_tokens)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_tokens)}")
print(f"Test tokens: {len(test_tokens)}")

Loading Thomas Jefferson speeches...


data/state_ofthe_union_texts.jsonl:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/219 [00:00<?, ? examples/s]

Vocab size: 15031
Train tokens: 279502
Test tokens: 31056


In [10]:
train_idx = [word2idx[w] for w in train_tokens]
test_idx = [word2idx[w] for w in test_tokens if w in word2idx]

count_model = CountTrigramModel(vocab_size=vocab_size, add_k=0.1)
count_model.train(train_idx)

ppl_count = calculate_perplexity_count(count_model, test_idx)
print(f"Count Model Test Perplexity: {ppl_count:.4f}")

Count Model Test Perplexity: 7788.0915


In [13]:
train_dataset = TrigramDataset(train_tokens, word2idx)
val_split = int(len(train_dataset) * 0.9)
train_ds, val_ds = torch.utils.data.random_split(train_dataset, [val_split, len(train_dataset) - val_split])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nn_model = NeuralTrigramModel(vocab_size=vocab_size, embed_size=64, hidden_size=128)
nn_model = train_neural_model(nn_model, train_loader, val_loader, epochs=50, patience=5, lr=1e-3, device=device)

test_dataset = TrigramDataset([w for w in test_tokens if w in word2idx], word2idx)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
ppl_nn = calculate_perplexity_neural(nn_model, test_loader, device=device)
print(f"Neural Model Test Perplexity: {ppl_nn:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ks = [0.001, 0.01, 0.1, 0.5, 1.0]
perplexities = []

for k in ks:
    m = CountTrigramModel(vocab_size=vocab_size, add_k=k)
    m.train(train_idx)
    ppl = calculate_perplexity_count(m, test_idx)
    perplexities.append(ppl)
    print(f"k={k:.3f}, Perplexity: {ppl:.2f}")

plt.figure(figsize=(10, 6))
plt.plot(ks, perplexities, marker='o', linestyle='-', color='#6366f1', linewidth=2.5)
plt.xscale('log')
plt.xlabel('Smoothing Parameter (k)', fontsize=12)
plt.ylabel('Test Perplexity', fontsize=12)
plt.title('Ablation Study: smoothing effect on perplexity', fontsize=14)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [14]:
# Save Neural Model
torch.save(nn_model.state_dict(), 'nn_model.pth')

# Save Count Model (using pickle to preserve defaultdict structure)
with open('count_model.pkl', 'wb') as f:
    pickle.dump(count_model, f)

# Save Vocab
with open('vocab.json', 'w') as f:
    json.dump({'word2idx': word2idx, 'idx2word': {str(k): v for k, v in idx2word.items()}}, f)

print("Models and vocab saved locally.")

Models and vocab saved locally.


In [15]:
repo_id = "bdanko/n-gram-modeling"
api = HfApi()

files_to_upload = ["nn_model.pth", "count_model.pkl", "vocab.json"]

for filename in files_to_upload:
    try:
        api.upload_file(
            path_or_fileobj=filename,
            path_in_repo=filename,
            repo_id=repo_id
        )
        print(f"Uploaded {filename} to {repo_id}")
    except Exception as e:
        print(f"Failed to upload {filename}: {e}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  nn_model.pth                :   5%|4         |  581kB / 11.7MB            

Uploaded nn_model.pth to bdanko/n-gram-modeling


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  count_model.pkl             :  91%|#########1| 3.64MB / 3.99MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded count_model.pkl to bdanko/n-gram-modeling
Uploaded vocab.json to bdanko/n-gram-modeling


In [ ]:
hf_nn_path = hf_hub_download(repo_id=repo_id, filename="nn_model.pth")
hf_count_path = hf_hub_download(repo_id=repo_id, filename="count_model.pkl")
hf_vocab_path = hf_hub_download(repo_id=repo_id, filename="vocab.json")

with open(hf_vocab_path, 'r') as f:
    hf_vocab_data = json.load(f)
    hf_word2idx = hf_vocab_data['word2idx']
    hf_idx2word = {int(k): v for k, v in hf_vocab_data['idx2word'].items()}

# Load Neural Model
loaded_nn_model = NeuralTrigramModel(vocab_size=len(hf_word2idx))
loaded_nn_model.load_state_dict(torch.load(hf_nn_path, weights_only=True))
loaded_nn_model.eval()

# Load Count Model
with open(hf_count_path, 'rb') as f:
    loaded_count_model = pickle.load(f)

print("Models loaded successfully from HF.")

nn_model.pth:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

count_model.pkl:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Models loaded successfully from HF.


In [ ]:
w1_word, w2_word = "another", "year"
w1, w2 = hf_word2idx[w1_word], hf_word2idx[w2_word]
seed_text = f"{w1_word} {w2_word}"

models_to_test = [("Count-based Trigram Model", loaded_count_model), ("Neural Trigram Model", loaded_nn_model)]

for name, model in models_to_test:
    print(f"\nEvaluating {name}")

    print("\nGreedy")
    print(generate_greedy(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80))

    print("\nBeam Search (width=3)")
    print(generate_beam_search(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, beam_width=3))

    print("\nTop-K Sampling (K=5)")
    print(generate_top_k(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, k=5))

    print("\nNucleus Sampling (p=0.9)")
    print(generate_nucleus(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, p=0.9))


Evaluating Count-based Trigram Model

Greedy
fellow citizens in the treasury on the part of the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the united states and the

Beam Search (width=3)
fellow citizens in the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and of the united states and

Top-K Sampling (K=5)
fellow citizens in cuba and puerto rico and an